# 9 — The model zoo

Notebooks 00–08 built the machine one mechanism at a time. `tensorlib.zoo` is
the machine used in anger: a spanning set of real architectures — GPT-2, a
Llama block, attention variants, two PDEs — each written as an ordinary L0
program.

The thesis is one sentence: **every entry is checked against an independent
numpy denotation**, so the zoo is what keeps the rest of the library honest.
The library is white-box and unusual; a bug in the layout algebra, the AD
transform, or the marker DSL would show up as a program that runs but
disagrees with the plain-numpy reference an outsider would write. Each
`ZooModel` carries that reference (`ref`) and the dim order to compare on
(`order`), and re-running the zoo is the regression suite.

A second job (LEVELS.md): the entries are the **test corpus for every level
above L0**. Peak memory (notebook 10), then DCE, checkpointing, sharding,
fusion — each will be measured on these exact programs. They were chosen to
span *mechanisms* (structured heads, RoPE pairs, GQA repeats, online-softmax,
guards-as-boundaries, charts-as-staggering), not to crawl an architecture
gallery.

Parameters are not special here: weights and activations are both just
`input` tensors. At L0 there is no notion of "the model" versus "the batch" —
that distinction is a later level's concern.

In [1]:
import nbhelp  # noqa: F401  — puts tensorlib on sys.path
import numpy as np
from tensorlib import Build
from tensorlib.autodiff import grad, numeric_grad
from tensorlib.ir import Instr, Program, run
from tensorlib.mdsl import Arg, Const
from tensorlib.zoo import (
    fdtd1d_staggered,
    flash_attention,
    gpt2,
    heat2d,
    llama_block,
)
from tensorlib.zoo.attention import flashsm

In [2]:
def check(zm, name):
    # Run the IR program and compare the output to the model's numpy ref.
    env = run(zm.program, zm.inputs)
    got = env[zm.out].to_numpy(order=zm.order)
    want = zm.ref(zm.numpy_inputs())
    print(f"  {name:<15}{zm.out:<8}{str(got.shape):<9} max|Δ| vs numpy = {np.abs(got - want).max():.2e}")


def sum_loss(zm):
    # Append a scalar loss (sum of the output) so we can take gradients.
    tail = Instr("loss", "reduce", (zm.out,), {"f": "sum", "dims": zm.order})
    return Program(tuple(zm.program.instrs) + (tail,))


_INFIX = {"add": "+", "sub": "-", "mul": "*", "div": "/"}


def expr(n, names):
    # Render a marker tree as infix — the same renderer notebook 07 used.
    if isinstance(n, Arg):
        return names[n.index] if n.index < len(names) else f"a{n.index}"
    if isinstance(n, Const):
        return f"{n.value}"
    if n.op in _INFIX:
        return f"({expr(n.args[0], names)} {_INFIX[n.op]} {expr(n.args[1], names)})"
    return f"{n.op}({', '.join(expr(a, names) for a in n.args)})"

## The `Build` helper — name management, deliberately not a frontend

Zoo programs are multi-layer and hand-written, so `build.py` exists to take
the tedium out of writing SSA by hand: `input` declares a leaf, `emit`/`pw`/
`red`/`repeat`/`const` append an instruction and hand back a **fresh unique
name**. That is *all* it does. There is no tracing, no operator overloading,
no expression tree — those live behind the marker DSL's stability boundary
(notebook 07). A `Build` is a topological-order transcription aid, not a
language.

Watch the name manager: emit `mul` twice and the second becomes `mul1`, so a
builder can name by intent and never collide.

In [3]:
b = Build()
x = b.input("x")
w = b.input("w")
p = b.pw("mul", x, w)  # auto-named from the op
p2 = b.pw("mul", p, p)  # 'mul' is taken -> 'mul1'
y = b.red("sum", p2, ("i",))
print(b.program())
print()
print("names it chose:", p, p2, y, "  (collision resolved, no hand-numbering)")
print("Build is ~50 lines: input / emit / pw / red / repeat / bcast / const / program")

x = input()
w = input()
mul = pointwise(x, w; f='mul')
mul1 = pointwise(mul, mul; f='mul')
sum = reduce(mul1; f='sum', dims=('i',))

names it chose: mul mul1 sum   (collision resolved, no hand-numbering)
Build is ~50 lines: input / emit / pw / red / repeat / bcast / const / program


## GPT-2: the program is ordinary IR

`gpt2()` is LayerNorm + causal multi-head attention + GELU MLP + residuals,
the baseline canon. Nothing about it is a special "model" object — it is a
`Program` of the same instructions notebooks 02–05 used. Here are its first
few, and its op vocabulary: every op is one you have already met.

In [4]:
m = gpt2()
print(f"gpt2: {len(m.program.instrs)} instructions, {len(m.inputs)} input tensors")
print()
for i in m.program.instrs[:9]:
    print("  ", i)
print("   ...")
print()
print("op vocabulary:", sorted({i.op for i in m.program.instrs}))

gpt2: 218 instructions, 29 input tensors

   x = input()
   pos = input()
   h0 = pointwise(x, pos; f='add')
   L0.ln1g = input()
   L0.ln1b = input()
   L0.wq = input()
   L0.wk = input()
   L0.wv = input()
   L0.wo = input()
   ...

op vocabulary: ['const', 'input', 'iota', 'pointwise', 'reduce', 'rename', 'repeat']


### Heads are born as dims — never as splits

The one design commitment worth staring at: `wq` is shaped `(d, nh, hk)` —
model width, then **head** and **head-width** as first-class dims of the
weight. So the query projection is a single contraction over `d` that leaves
`(t, nh, hk)` standing; the heads were never manufactured by splitting a flat
`(nh·hk)` axis (D5, names-first). Consequently **`split` never appears** in
the whole program — splitting is a *lowering* choice for a later level, not
part of the denotation. And the program agrees with numpy to machine
precision.

In [5]:
for nm in ("L0.wq", "L0.wk", "L0.wv", "L0.wo", "L0.w1"):
    t = m.inputs[nm]
    print(f"  {nm:<8} dims {t.names}  shape {t.to_numpy().shape}")
print()
print("'split' in the program:", "split" in {i.op for i in m.program.instrs})
print()
check(m, "gpt2")

  L0.wq    dims ('d', 'nh', 'hk')  shape (6, 2, 3)
  L0.wk    dims ('d', 'nh', 'hk')  shape (6, 2, 3)
  L0.wv    dims ('d', 'nh', 'hk')  shape (6, 2, 3)
  L0.wo    dims ('nh', 'hk', 'd')  shape (2, 3, 6)
  L0.w1    dims ('d', 'm')  shape (6, 8)

'split' in the program: False

  gpt2           logits  (4, 5)    max|Δ| vs numpy = 4.44e-16


## Llama block: RoPE without splits, GQA by declaration

`llama_block()` swaps in RMSNorm, RoPE, grouped-query attention and SwiGLU.
Two of those are where a conventional tensor library reaches for reshapes and
interleaves; here they are just structured dims and repeats.

Look at the weight shapes. `wq` is `(d, g, r, c, u)`: `g` = kv-group, `r` =
query-head-within-group, `c` = which rotary pair, and `u` = the {real, imag}
slot of that pair. The rotary pair structure is **born in the weight** — so
rotation never has to carve a head vector into even/odd halves.

In [6]:
lm = llama_block()
print(f"llama_block: {len(lm.program.instrs)} instructions")
for nm in ("wq", "wk", "wv", "wo", "omega"):
    t = lm.inputs[nm]
    print(f"  {nm:<6} dims {t.names}  shape {t.to_numpy().shape}")
print("            (u = the {re, im} slot of each rotary pair — a dim, not a slice)")
print()
check(lm, "llama_block")

llama_block: 127 instructions
  wq     dims ('d', 'g', 'r', 'c', 'u')  shape (6, 2, 2, 2, 2)
  wk     dims ('d', 'g', 'c', 'u')  shape (6, 2, 2, 2)
  wv     dims ('d', 'g', 'kv')  shape (6, 2, 3)
  wo     dims ('g', 'r', 'kv', 'd')  shape (2, 2, 3, 6)
  omega  dims ('c',)  shape (2,)
            (u = the {re, im} slot of each rotary pair — a dim, not a slice)

  llama_block    out     (4, 6)    max|Δ| vs numpy = 2.22e-16


### RoPE = selects + trig; the score is a sum of two u-slot contractions

Rotating a pair `(q0, q1)` by angle θ is textbook — `q0·cos − q1·sin`,
`q0·sin + q1·cos`. Because the `u` slot is a dim, `q0` and `q1` are
`select(u=0)` / `select(u=1)` — zero-byte views, not gathers — and the angle
comes from an `iota` over the position dim times `omega`. No concatenation,
no interleave.

The dot product `Re(q̄ · k)` over a complex head then falls out as the sum of
the two real-slot contractions `sc0 + sc1`, each an ordinary `reduce(sum)`
over the pair index `c`.

In [7]:
want = {"pos", "theta", "cos", "sin", "q0", "q1", "rot0", "rot1", "sc0", "sc1"}
for i in lm.program.instrs:
    if i.var in want:
        print("  ", i)

   pos = iota(ot; name='t')
   theta = pointwise(pos, ot; f='mul')
   cos = pointwise(theta; f='cos')
   sin = pointwise(theta; f='sin')
   q0 = select(q; coords={'u': 0})
   q1 = select(q; coords={'u': 1})
   rot0 = pointwise(mul3, mul4; f='sub')
   rot1 = pointwise(mul5, mul6; f='add')
   sc0 = reduce(mul11; f='sum', dims=('c',))
   sc1 = reduce(mul12; f='sum', dims=('c',))


### GQA is repeat-by-declaration

Grouped-query attention shares one K/V head across a group of query heads.
Here that is not a broadcast trick bolted on afterward — it falls out of the
dims. Query activations carry both `g` and `r`; the K projection `wk` carries
only `g`. When the score contraction lines them up, the missing `r` on K is
supplied by a `repeat` — the KV head declared identical across the `r` query
heads in its group. GQA *is* the repeat.

In [8]:
print("wq dims:", lm.inputs["wq"].names, " (query heads live on g AND r)")
print("wk dims:", lm.inputs["wk"].names, " (kv heads live on g only)")
print()
reps = [i for i in lm.program.instrs if i.op == "repeat" and i.params.get("name") == "r"]
print(f"{len(reps)} repeat(...; name='r') instructions supply r to the kv side, e.g.")
for i in reps[:3]:
    print("  ", i)

wq dims: ('d', 'g', 'r', 'c', 'u')  (query heads live on g AND r)
wk dims: ('d', 'g', 'c', 'u')  (kv heads live on g only)

6 repeat(...; name='r') instructions supply r to the kv side, e.g.
   rep3 = repeat(rep2; name='r', extent=(0, 2))
   rep12 = repeat(rep11; name='r', extent=(0, 2))
   rep14 = repeat(rep13; name='r', extent=(0, 2))


## Flash attention: the online-softmax defreducer

`flash_attention()` is the flagship — the one that ties the marker DSL to the
fusion story two levels up. Online softmax carries a three-part running state
`(m, ℓ, o)`: the running max, the running denominator, and the running
weighted sum of values. Merging two partial states rescales both to their
joint max. That combine is **associative** (the online-softmax lemma), so it
is exactly a `defreducer` — declared once, below.

In [9]:
L = ("mL", "denL", "oL")
R = ("mR", "denR", "oR")
print("state =", flashsm.state, " element =", flashsm.element)
print("init  =", flashsm.init, "  (the monoid identity; exp(-1e30 - m) underflows to 0)")
print("lift(s, v) =", tuple(expr(n, ("s", "v")) for n in flashsm.lift))
print("project(m, den, o) =", expr(flashsm.project, ("m", "den", "o")))
print("combine:")
for name, node in zip(("m'", "den'", "o'"), flashsm.combine):
    print(f"    {name:<5}=", expr(node, L + R))

state = 3  element = 2
init  = (-1e+30, 0.0, 0.0)   (the monoid identity; exp(-1e30 - m) underflows to 0)
lift(s, v) = ('s', '1.0', 'v')
project(m, den, o) = (o / den)
combine:
    m'   = maximum(mL, mR)
    den' = ((denL * exp((mL - maximum(mL, mR)))) + (denR * exp((mR - maximum(mL, mR)))))
    o'   = ((oL * exp((mL - maximum(mL, mR)))) + (oR * exp((mR - maximum(mL, mR)))))


The flash program feeds masked scores and values straight into that reducer —
a single `reduce(f='zoo.flashsm', dims=('s',))` where the naive program
materializes a full softmax then contracts. Same denotation, different
program: the pair is the before/after of the fusion argument. First, forward:
`naive=True` builds the materialized version, `naive=False` the fused one, and
both match the reference.

In [10]:
fa = flash_attention(naive=False)
na = flash_attention(naive=True)
print("flash program:", len(fa.program.instrs), "instructions; the accumulator is one op:")
for i in fa.program.instrs:
    if i.op == "reduce" and i.params.get("f") == "zoo.flashsm":
        print("  ", i)
print("naive program:", len(na.program.instrs), "instructions (softmax then contract)")
print()
check(fa, "flash")
check(na, "naive")

flash program: 15 instructions; the accumulator is one op:
   flash = reduce(se, ve; f='zoo.flashsm', dims=('s',))
naive program: 23 instructions (softmax then contract)

  flash          flash   (5, 2)    max|Δ| vs numpy = 1.11e-16
  naive          ctx     (5, 2)    max|Δ| vs numpy = 2.78e-17


### The backward pass is DERIVED — no hand-written flash rule

Here is the payoff. Nobody wrote a backward rule for flash attention. The
combine is a marker tree, so `grad` differentiates it by the generic
composite-reducer BPTT machinery from notebook 07 — the same machine that
handled `linrec`. Below: the gradients of the fused program agree with the
gradients of the naive program (and with finite differences) to machine
precision, and the joint program is full of `zoo.flashsm.adj*` / `.s*`
reducers that were generated, not authored.

In [11]:
for wrt in ("q", "k", "v"):
    jf, gf = grad(sum_loss(fa), "loss", fa.inputs)
    ef = run(jf, fa.inputs)[gf[wrt]].to_numpy(order=fa.inputs[wrt].names)
    jn, gn = grad(sum_loss(na), "loss", na.inputs)
    en = run(jn, na.inputs)[gn[wrt]].to_numpy(order=na.inputs[wrt].names)
    nd = numeric_grad(sum_loss(na), "loss", wrt, na.inputs)
    print(f"  dL/d{wrt}:  flash vs naive {np.abs(ef - en).max():.2e}   flash vs finite-diff {np.abs(ef - nd).max():.2e}")

print()
generated = sorted({i.params["f"] for i in jf.instrs if i.op in ("reduce", "scan") and "flashsm" in i.params.get("f", "")})
print("reducers the backward pass generated for flash:")
print("  ", generated)
print("  (.adjN = derived backward recurrences, .sN = forward-state re-scanners — none hand-written)")

  dL/dq:  flash vs naive 2.22e-16   flash vs finite-diff 4.06e-10


  dL/dk:  flash vs naive 8.33e-17   flash vs finite-diff 5.74e-10

  dL/dv:  flash vs naive 5.55e-17   flash vs finite-diff 3.16e-10

reducers the backward pass generated for flash:
   ['zoo.flashsm', 'zoo.flashsm.adj0', 'zoo.flashsm.adj1', 'zoo.flashsm.adj2', 'zoo.flashsm.s0', 'zoo.flashsm.s1', 'zoo.flashsm.s2']
  (.adjN = derived backward recurrences, .sN = forward-state re-scanners — none hand-written)


## Physics — heat2d: Dirichlet ghosts are pad guards

The two physics entries exercise machinery transformers never touch. `heat2d`
is explicit-Euler diffusion. Every neighbor access `u[i±1]` at the boundary
needs a value from outside the grid; the Dirichlet-0 condition says that value
is zero. In this library that boundary condition *is* a guard: each ghost
access is `shift → slice → pad(0)`, so the physics of the boundary is spelled
out in layout ops (notebook 04), not patched in by an if-statement. The step
is a `fold` (notebook 08), and the whole thing matches the padded-numpy
reference.

In [12]:
hm = heat2d()
uf = hm.program.instr(hm.out)
step = uf.params["step"]
print("heat2d: a fold whose step is a", len(step.instrs), "-instruction program.")
print("one ghost access (u[i-1] with a zero ghost) is exactly shift -> slice -> pad(0):")
for i in step.instrs:
    if i.var in ("sh", "sl", "gh"):
        print("  ", i)
print()
check(hm, "heat2d")

heat2d: a fold whose step is a 22 -instruction program.
one ghost access (u[i-1] with a zero ghost) is exactly shift -> slice -> pad(0):
   sh = shift(u; deltas={'x': 1})
   sl = slice(sh; ranges={'x': (1, 5)})
   gh = pad(sl; fill=0.0, extents={'x': (0, 5)})

  heat2d         uf      (5, 5)    max|Δ| vs numpy = 2.22e-16


## FDTD staggered — exact half-integer charts

`fdtd1d_staggered` is a 1D Yee-grid leapfrog: the electric field E lives at
integer positions, the magnetic field H at **half-integer** positions. That
staggering is carried by charts with exact rational origins — `Fraction(1, 2)`,
not a rounded 0.5 — so the grid geometry is part of the tensor's identity.

In [13]:
fd = fdtd1d_staggered()
print("E0 lives on chart:", fd.inputs["E0"].layout.dims[0].chart)
print("H0 lives on chart:", fd.inputs["H0"].layout.dims[0].chart, "  (exact Fraction(1,2) origin)")
print()
check(fd, "fdtd1d")

E0 lives on chart: pos[x: 0 step 1]
H0 lives on chart: pos[x: 0.5 step 1]   (exact Fraction(1,2) origin)

  fdtd1d         Ef      (6,)      max|Δ| vs numpy = 0.00e+00


### Differencing E misaligns with H — until `with_charts` says otherwise

This is the discretization's honesty made syntax. The update `H ← H + c·(E[i+1]
− E[i])` differences E — and `E[i+1] − E[i]` physically lives at the *half*
step, on the H grid, not the E grid. The alignment check (D17) sees two
frames offset by a step and **refuses to subtract them**: the shift moved the
chart frame, and the library will not silently paper over it.

The fix is not a shift — it is a *declaration*. `with_charts` says "this
difference lives at i+½", moving both operands onto the H grid, and then the
subtraction is well-posed. Below: the naive step (no recharting) is refused;
the recharted step succeeds and lands the difference on the H grid.

In [14]:
from tensorlib import Tensor  # noqa: E402
from tensorlib.chart import chart  # noqa: E402
from fractions import Fraction  # noqa: E402

N = 6
e_chart = chart(0, 1, axis="x")
h_chart = chart(Fraction(1, 2), 1, axis="x")
Efield = Tensor.from_numpy(np.arange(N, dtype=np.float64), ("x",)).with_charts(x=e_chart)
Hfield = Tensor.from_numpy(np.zeros(N - 1), ("x",)).with_charts(x=h_chart)


def diff_E_step(rechart):
    sb = Build()
    E = sb.input("E")
    Ea = sb.emit("slice", (sb.emit("shift", (E,), hint="Es", deltas={"x": -1}),), hint="Ea", ranges={"x": (0, N - 1)})
    Eb = sb.emit("slice", (E,), hint="Eb", ranges={"x": (0, N - 1)})
    if rechart:
        Ea = sb.emit("with_charts", (Ea,), hint="Ea_h", charts={"x": h_chart})
        Eb = sb.emit("with_charts", (Eb,), hint="Eb_h", charts={"x": h_chart})
    sb.pw("sub", Ea, Eb, hint="dE")
    return sb.program()


try:
    run(diff_E_step(rechart=False), {"E": Efield})
except ValueError as e:
    print("refused (no recharting):")
    print("  ", str(e).replace("\n", "\n   "))

print()
env = run(diff_E_step(rechart=True), {"E": Efield})
dE = env["dE"]
print("with_charts declares the half-step:  dE =", dE.to_numpy(), " on chart", dE.layout.dims[0].chart)

refused (no recharting):
   pointwise(sub) requires aligned operands:
     operand 1, dim 'x': frames offset by -1 steps  ->  shift(x=-1)

with_charts declares the half-step:  dE = [1. 1. 1. 1. 1.]  on chart pos[x: 0.5 step 1]


The suggested repair the diagnosis prints (`shift(x=-1)`) is the *mechanical*
way to re-align two integer frames; the *physical* repair is the recharting,
which says the value moved to the staggered grid. The zoo entry does exactly
this recharting at each half-step — and, because the charts are honest, the
**gradients come back on the right grids**: dL/dE0 on the integer grid, dL/dH0
on the half-integer grid, both matching finite differences.

In [15]:
prog = sum_loss(fd)
for wrt in ("E0", "H0"):
    jp, gr = grad(prog, "loss", fd.inputs)
    g = run(jp, fd.inputs)[gr[wrt]]
    nd = numeric_grad(prog, "loss", wrt, fd.inputs)
    print(f"  dL/d{wrt}: chart {g.layout.dims[0].chart!s:<22} vs finite-diff max|Δ| = {np.abs(g.to_numpy() - nd).max():.2e}")

  dL/dE0: chart pos[x: 0 step 1]       vs finite-diff max|Δ| = 1.64e-10
  dL/dH0: chart pos[x: 0.5 step 1]     vs finite-diff max|Δ| = 8.74e-11


## The recorded boundaries — and the entries as benchmarks

The zoo is a *spanning* set, and its edges are drawn on purpose. Three
mechanisms are deliberately left out (LEVELS.md), each because it breaks the
static-shape / no-data-dependence contract L0 is built on:

- **MoE routing / top-k** — a data-dependent gather (which expert each token
  goes to is known only at runtime);
- **KV-cache decode** — mutation of a growing buffer, not a pure recurrence;
- **dynamic shapes** — sequence lengths that are values, not constants.

These are not "not yet implemented" so much as "the next representation's
problem": each needs machinery (dynamic gather, in-place mutation, symbolic
extents) that would change what L0 *means*, so they wait for the level that
introduces it honestly.

Meanwhile every entry above doubles as a **benchmark for every level up the
ladder**. Notebook 10 runs the peak-memory simulator on exactly these
programs; later levels re-run them for DCE, checkpointing, sharding, and
fusion. A model that is *data* — an inspectable `Program` plus a numpy oracle
— is a model you can measure, transform, and re-check at each step.

---
Known gaps (CONCERNS #26). The zoo's numeric hygiene is toy-scale on purpose:
`-1e9` masking rather than `-inf`, tiny widths, float64 throughout, no
dropout, and the flash reducer's `-1e30` init leans on `exp` underflow for its
identity. That is adequate for denotation tests and cost-model corpora — the
job here — but it is not a numerics benchmark. And `reduce(max)` tie-splitting
is reachable through the flash combine's `maximum` partials if two scores tie
*exactly*; the tests use continuous random scores, so it never fires, but on
quantized inputs it would need the sub-gradient convention pinned down.